<a href="https://colab.research.google.com/github/sourabhverma1302/AI-ML-Journey/blob/main/Wine_Class_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ========================== DATA PREPARATION ==========================
wine = load_wine()
X = wine.data
y = wine.target

print("=== Wine Dataset Info ===")
print(f"Features shape: {X.shape}")
print(f"Number of classes: {len(set(y))}")
print(f"Class names: {wine.target_names}\n")

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X = torch.FloatTensor(X_scaled)
y = torch.LongTensor(y)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ========================== MODEL ==========================
model = nn.Sequential(
    nn.Linear(13, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 3)
)

# ========================== TRAINING SETUP ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=150, gamma=0.5)

# ========================== TRAINING LOOP ==========================
print("Training Started...\n")

for epoch in range(600):
    model.train()                    # Enable Dropout & BatchNorm
    optimizer.zero_grad()

    output = model(X_train)
    loss = criterion(output, y_train)

    loss.backward()
    optimizer.step()
    scheduler.step()                 # Update learning rate

    # Print progress
    if epoch % 100 == 0:
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f} | LR: {current_lr:.5f}')

# ========================== EVALUATION ==========================
model.eval()                         # Disable Dropout
with torch.no_grad():
    test_output = model(X_test)
    predictions = test_output.argmax(dim=1)

    test_accuracy = (predictions == y_test).float().mean()
    print(f'\nFinal Test Accuracy: {test_accuracy * 100:.2f}%')

=== Wine Dataset Info ===
Features shape: (178, 13)
Number of classes: 3
Class names: ['class_0' 'class_1' 'class_2']

Training Started...

Epoch   0 | Loss: 1.0746 | LR: 0.00100
Epoch 100 | Loss: 0.0973 | LR: 0.00100
Epoch 200 | Loss: 0.0205 | LR: 0.00050
Epoch 300 | Loss: 0.0199 | LR: 0.00025
Epoch 400 | Loss: 0.0105 | LR: 0.00025
Epoch 500 | Loss: 0.0097 | LR: 0.00013

Final Test Accuracy: 100.00%
